# 03 · 2-CTA MMA and multicast

`fp16_gemm_1` pairs two CTAs into a 2×1 cluster that issue one 2-CTA MMA together. This halves the B-tile smem per CTA, so more pipeline stages fit, so more DRAM latency is hidden. TMA multicast loads each tile once and broadcasts to both CTAs, cutting L2 read traffic.

Every clause of that is true. The kernel is still **slower** than the simpler one at the shape we have been using.

## 1. The question

The 2-CTA design buys pipeline depth and pays cluster synchronisation.

| | 1-CTA MMA | 2-CTA MMA |
|---|---|---|
| Instruction shape | (128, 256, 16) | (256, 256, 16) |
| Per-CTA shape | (128, 256, 16) | (128, 128, 16) |
| A/B pipeline stages | 4 | **7** |

**The question:** when does the deeper pipeline earn back the synchronisation cost? This is not a yes/no about the optimisation — it is about finding its *regime*.

### Setup

Every notebook starts here. If this machine cannot profile, the collection cells below print their command instead of failing, and the notebook still reads end to end.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "tools"))
import check_env, nbhelp, trace_summary as ts

env = check_env.report()

Environment
----------------------------------------------------------
  GPU           NVIDIA B300 SXM6 AC  [SM103 (Blackwell)]
  Driver        595.58.03
  cutlass-dsl   4.7.1
  iket API      yes
  run-iket      yes
----------------------------------------------------------
  Live profiling: AVAILABLE


## 2. The instrumentation

Same range vocabulary as notebook 02, so the two kernels are directly comparable. The point of keeping instrumentation identical across variants is exactly this: it makes A/B a subtraction rather than an interpretation.

In [2]:
nbhelp.show_iket_calls(
    nbhelp.kernel("instrumented", "fp16_gemm_1x_iket_detailed.py"),
    limit=10,
)

fp16_gemm_1x_iket_detailed.py: 32 IKET call sites (showing first 10)
------------------------------------------------------------------------
>  113      cute.experimental.iket.range_push("prologue")
       ...
>  283      cute.experimental.iket.range_pop()
       ...
>  288      cute.experimental.iket.range_push("mainloop")
       ...
>  296          cute.experimental.iket.range_push("prefetch")
       ...
>  298              cute.experimental.iket.range_push("prefetch_iter")
       ...
>  300              cute.experimental.iket.range_push("prod_acquire")
       ...
>  302              cute.experimental.iket.range_pop()
       ...
>  305              cute.experimental.iket.range_push("tma_issue")
       ...
>  320              cute.experimental.iket.range_pop()
>  321              cute.experimental.iket.range_pop()


## 3. Run it

Two shapes, because the answer depends on K:

In [2]:
for tag, mnk in [("k512", "4096,4096,512"), ("k4096", "4096,4096,4096")]:
    cmd = (
        f"run-iket --output-dir out/nb03_{tag} --clobber profile --postprocess all -- "
        f"python {nbhelp.kernel('instrumented', 'fp16_gemm_1x_iket_detailed.py')} --mnk {mnk}"
    )
    print(f"===== {tag} =====")
    check_env.run_or_show(cmd, env, est="40 s (K=512), 2 min (K=4096)")

===== k512 =====
$ run-iket --output-dir out/nb03_k512 --clobber profile --postprocess all -- python /path/to/cutedsl-iket-tutorial-notebooks --mnk 4096,4096,512

[run-iket] ============= Dry run application first to collect some info... =============
Running Blackwell fp16 GEMM example 1 with:
  mnk:       [4096, 4096, 512]
  tolerance: 0.1

PASS
[run-iket] Auto-computed context buffer size: 0.00 GB (1,469,644 bytes)
[run-iket] ============= Run application for real profiling... =============
Running Blackwell fp16 GEMM example 1 with:
  mnk:       [4096, 4096, 512]
  tolerance: 0.1

PASS
[run-iket] Dumped perfetto trace to /path/to/cutedsl-iket-tutorial-notebooks
[run-iket] Dumped json trace to /path/to/cutedsl-iket-tutorial-notebooks
[run-iket] Dumped compressed trace to /path/to/cutedsl-iket-tutorial-notebooks
[run-iket] Dumped perfetto HTML viewer to /path/to/cutedsl-iket-tutorial-notebooks
[run-iket] To view: cd /path/to/cutedsl-iket-tutorial-notebooks && python3 -m http.server 8

## 4. The trace

The defining visual: both CTAs of the cluster advance in lockstep - top: leader CTA, bottom: non-leader CTA. That is the 2-CTA MMA working — and also the coupling you are paying for.

In [4]:
nbhelp.figure("nb03/gemm1_2cta_lockstep_perfetto.png", "Two CTAs advancing in lockstep through the mainloop.")

In [5]:
nbhelp.figure("nb03/gemm1_large_k_crossover_a.png", "At large K the picture changes.")

In [6]:
try:
    trace = ts.load(ts.find_trace("out/nb03_k4096"))
    ts.print_summary(ts.summarize(trace), top=10)
except FileNotFoundError:
    print("No local trace here — the reference numbers below were measured on a NVIDIA B300.")

kernel_cutlass_kernel_TiledMMA_ThrLayoutVMNK21111000_PermutationMNK____MMAAtom_ThrID21_ShapeMNK25625616_TVLayoutA2128161281256_TVLayoutB2128161281256_TVLayoutC21282561281256_cutlasscutenv_0   grid 32x16x1  block 128x1x1  warps traced 2048
warp-lifetime span: 92.0 us

range                      count      median         p90       total
--------------------------------------------------------------------
epilogue                    2048     21.184u     22.080u   34847.04u
acc_wait                    2048     19.136u     19.808u   30361.31u
mainloop                    2048      0.032u     18.208u    9322.78u
k_tile_iter                32768      0.256u      0.288u    8338.43u
prod_acquire               32768      0.096u      0.192u    4345.41u
prologue                    2048      0.624u      0.704u    1288.93u
epi_subtile                 8192      0.128u      0.160u    1000.96u
acc_release                 2048      0.352u      0.416u     738.27u
cons_wait                  16384      0.03

## 5. The finding

Measured on a NVIDIA B300, warp-lifetime span, against notebook 02's PS2 as the baseline:

| Shape | gemm 0x (1-CTA) | gemm 1x (2-CTA) | Verdict |
|---|---|---|---|
| `mnk=4096,4096,512` | **26.18 µs** | 28.70 µs | 2-CTA is **10% slower** |
| `mnk=4096,4096,4096` | 113.31 µs | **92.32 µs** | 2-CTA is **23% faster** |


**The optimisation has a regime, and K is the axis.**

At K=512 the mainloop is short. There are few K-tiles over which to amortise the cluster synchronisation, and the deeper pipeline never fills enough to matter. The extra stages are dead weight.

At K=4096 the mainloop dominates. Now the 7-stage pipeline has something to hide, and the per-iteration numbers show it: `k_tile_iter` median 0.256 µs across 32,768 iterations, with `cons_wait` at just 0.032 µs. The MMA warp is almost never starved.

**Why this matters beyond this kernel:** a benchmark at one shape would have told you 2-CTA MMA was a regression. It is not. It is an optimisation whose benefit scales with K, and you cannot see that from a single measurement — you need two points and a reason for the difference.

*A caveat kept from the original study: multicast measured close to neutral here. Isolating its L2 benefit needs shapes that break L2 residency (M=N=16K), which this notebook does not cover.*

**Next:** [04 · Warp specialization](04_warp_specialization.ipynb)

---
*Measured on a single NVIDIA B300, public `nvidia-cutlass-dsl` 4.7.1. Your absolute numbers will differ; the shapes in the trace are the transferable part.*